In [ ]:
import os
import requests


GITHUB_TOKEN = os.getenv("GITHUB_TOKEN")

headers = {
    "Accept": "application/vnd.github+json",
    "Authorization": f"token {GITHUB_TOKEN}"
}

r = requests.get("https://api.github.com/rate_limit", headers=headers)

print("Status:", r.status_code)
print(r.json())

In [ ]:
import requests
import pandas as pd
import time
from datetime import datetime

import os
GITHUB_TOKEN = os.getenv("GITHUB_TOKEN")

headers = {
    "Accept": "application/vnd.github+json",

    "Authorization": f"token {GITHUB_TOKEN}"
}

repos_and_limits = {
    "microsoft/vscode": 400,
    "tensorflow/tensorflow": 300,
    "pytorch/pytorch": 300
}

all_issues = []

def fetch_issues(repo, limit):
    collected = 0
    page = 1

    while collected < limit:
        url = f"https://api.github.com/repos/{repo}/issues"
        params = {
          
            "state": "closed",
            "per_page": 100,
            "page": page
        }

        response = requests.get(url, headers=headers, params=params)
        print(f"{repo} | page {page} | status {response.status_code}")

        if response.status_code == 403:
            print("Rate limit hit. Sleeping...")
            time.sleep(60)
            continue

        if response.status_code != 200:
            print("Error:", response.text[:300])
            break

        issues = response.json()

        if not issues:
            break

        for issue in issues:
            if "pull_request" in issue:
                continue

            if issue.get("comments", 0) < 2:
                continue

            title = issue.get("title") or ""
            body = issue.get("body") or ""
            labels = [label["name"] for label in issue.get("labels", [])]

            created_at = issue.get("created_at")
            closed_at = issue.get("closed_at")

            resolution_time_hours = None
            if created_at and closed_at:
                t1 = datetime.strptime(created_at, "%Y-%m-%dT%H:%M:%SZ")
                t2 = datetime.strptime(closed_at, "%Y-%m-%dT%H:%M:%SZ")
                resolution_time_hours = (t2 - t1).total_seconds() / 3600

            all_issues.append({
                "repo": repo,
                "issue_id": issue.get("id"),
                "issue_number": issue.get("number"),
                "title": title,
                "body": body,
                "full_text": (title + " " + body).strip(),
                "labels": ", ".join(labels),
                "state": issue.get("state"),
                "created_at": created_at,
                "closed_at": closed_at,
                "resolution_time_hours": resolution_time_hours,
                "comments_count": issue.get("comments"),
                "author_login": (issue.get("user") or {}).get("login"),
                "author_type": (issue.get("user") or {}).get("type"),
                "assignee_login": (issue.get("assignee") or {}).get("login") if issue.get("assignee") else None
            })

            collected += 1
            if collected >= limit:
                break

        page += 1
        time.sleep(0.5)

for repo, limit in repos_and_limits.items():
    fetch_issues(repo, limit)

issues_df = pd.DataFrame(all_issues)
issues_df.to_csv("issues.csv", index=False)

print("Saved issues.csv")
print("Total rows:", len(issues_df))
issues_df.head()

In [ ]:
print(issues_df.shape)
print(issues_df.columns.tolist())
print(issues_df["repo"].value_counts())
print(issues_df["state"].value_counts())

In [ ]:
closed_issues_df = issues_df[issues_df["state"] == "closed"].copy()


all_comments = []

def fetch_comments(repo, issue_number):
    page = 1

    while True:
        url = f"https://api.github.com/repos/{repo}/issues/{issue_number}/comments"
        params = {
            "per_page": 100,
            "page": page
        }

        response = requests.get(url, headers=headers, params=params)
        print(f"comments | {repo} #{issue_number} | page {page} | status {response.status_code}")

 
        if response.status_code == 403:
            print("Rate limit hit... sleeping")
            time.sleep(60)
            continue

        if response.status_code != 200:
            print("Error:", response.text[:300])
            break

        comments = response.json()

        if not comments:
            break

        for c in comments:

            if (c.get("user") or {}).get("type") == "Bot":
                continue

            if not c.get("body"):
                continue

            all_comments.append({
                "repo": repo,
                "issue_number": issue_number,
                "comment_id": c.get("id"),
                "comment_body": c.get("body") or "",
                "comment_created_at": c.get("created_at"),
                "comment_updated_at": c.get("updated_at"),
                "comment_author_login": (c.get("user") or {}).get("login"),
                "comment_author_type": (c.get("user") or {}).get("type"),
                "author_association": c.get("author_association")
            })

        page += 1
        time.sleep(0.5)

for _, row in closed_issues_df.iterrows():
    fetch_comments(row["repo"], int(row["issue_number"]))

comments_df = pd.DataFrame(all_comments)
comments_df.to_csv("comments.csv", index=False)

print("Saved comments.csv")
print("Total comment rows:", len(comments_df))
comments_df.head()

In [ ]:
print(comments_df.shape)
print(comments_df.columns.tolist())
print(comments_df["repo"].value_counts())

In [ ]:
comments_per_issue = comments_df.groupby(["repo", "issue_number"]).size()

print(comments_per_issue.describe())

In [ ]:
print(comments_df["comment_author_type"].value_counts())

In [ ]:
print((comments_df["comment_body"].str.strip() == "").sum())

In [ ]:
comments_df.head()

In [ ]:
all_events = []

def fetch_events(repo, issue_number):
    page = 1

    while True:
        url = f"https://api.github.com/repos/{repo}/issues/{issue_number}/events"
        params = {
            "per_page": 100,
            "page": page
        }

        response = requests.get(url, headers=headers, params=params)
        print(f"events | {repo} #{issue_number} | page {page} | status {response.status_code}")

 
        if response.status_code == 403:
            print("Rate limit hit... sleeping")
            time.sleep(60)
            continue

        if response.status_code != 200:
            print("Error:", response.text[:300])
            break

        events = response.json()

        if not events:
            break

        for e in events:

   
            if e.get("event") not in [
                "assigned",
                "unassigned",
                "labeled",
                "unlabeled",
                "reopened",
                "closed"
            ]:
                continue

            all_events.append({
                "repo": repo,
                "issue_number": issue_number,
                "event_id": e.get("id"),
                "event_type": e.get("event"),
                "event_created_at": e.get("created_at"),
                "actor_login": (e.get("actor") or {}).get("login") if e.get("actor") else None,
                "label_name": (e.get("label") or {}).get("name") if e.get("label") else None,
                "assignee_login": (e.get("assignee") or {}).get("login") if e.get("assignee") else None
            })

        page += 1
        time.sleep(0.5)

for _, row in closed_issues_df.iterrows():
    fetch_events(row["repo"], int(row["issue_number"]))

events_df = pd.DataFrame(all_events)
events_df.to_csv("events.csv", index=False)

print("Saved events.csv")
print("Total event rows:", len(events_df))
events_df.head()

In [ ]:
print(events_df.shape)
print(events_df.columns.tolist())
print(events_df["repo"].value_counts())

In [ ]:
print(events_df["event_type"].value_counts())

In [ ]:
events_per_issue = events_df.groupby(["repo", "issue_number"]).size()

print(events_per_issue.describe())

In [ ]:
events_df.head()

In [ ]:
from google.colab import files

files.download("issues.csv")
files.download("comments.csv")
files.download("events.csv")

In [ ]:
print("ISSUES:", issues_df.shape)
print("COMMENTS:", comments_df.shape)
print("EVENTS:", events_df.shape)

print("\nIssue columns:", issues_df.columns.tolist())
print("\nComment columns:", comments_df.columns.tolist())
print("\nEvent columns:", events_df.columns.tolist())